# 1. Verificação dos dispositivos no barramento I²C

## Objetivo

Confirmar que as duas IMUs MPU-6050 conectadas ao ESP32 estão sendo reconhecidas no barramento I²C com endereços diferentes.

Neste teste, não serão coletados dados de aceleração ou rotação. O programa apenas procura dispositivos conectados ao barramento I²C e apresenta seus endereços no monitor serial.

## Hardware necessário

* 1 ESP32 DevKit v1;
* 2 módulos MPU-6050;
* Cabos jumper;
* Cabo USB para conectar o ESP32 ao computador.

## Montagem do hardware

Antes de fazer ou alterar qualquer conexão, desconecte o ESP32 da alimentação USB.

As duas IMUs devem compartilhar as conexões de alimentação e do barramento I²C:

| ESP32 DevKit v1 | MPU-6050 A | MPU-6050 B |
| --------------- | ---------- | ---------- |
| 3V3             | VCC        | VCC        |
| GND             | GND        | GND        |
| GPIO 21         | SDA        | SDA        |
| GPIO 22         | SCL        | SCL        |
| GND             | AD0        | —          |
| 3V3             | —          | AD0        |

Os pinos `INT` não são utilizados neste teste e devem permanecer desconectados.

### Endereçamento das IMUs

O pino `AD0` define o endereço I²C utilizado por cada MPU-6050:

* `AD0` conectado ao `GND`: endereço `0x68`;
* `AD0` conectado ao `3V3`: endereço `0x69`.

Assim, a IMU A deve utilizar o endereço `0x68`, enquanto a IMU B deve utilizar o endereço `0x69`.

As conexões `SDA` e `SCL` são compartilhadas pelas duas IMUs. Isso é permitido porque cada sensor possui um endereço diferente no barramento.

## Representação das conexões

```text
ESP32 GPIO 21 (SDA) ───┬── SDA da IMU A
                       └── SDA da IMU B

ESP32 GPIO 22 (SCL) ───┬── SCL da IMU A
                       └── SCL da IMU B

ESP32 3V3 ─────────────┬── VCC da IMU A
                       ├── VCC da IMU B
                       └── AD0 da IMU B

ESP32 GND ─────────────┬── GND da IMU A
                       ├── GND da IMU B
                       └── AD0 da IMU A
```

## Firmware utilizado

Utilize o arquivo:

```text
testes_unitarios/imu_serial/01_scanner_i2c/01_scanner_i2c.ino
```

O firmware configura o barramento com:

* `SDA`: GPIO 21;
* `SCL`: GPIO 22;
* frequência I²C: 100 kHz;
* comunicação serial: 115200 baud.

## Procedimento

1. Confira todas as conexões antes de alimentar o circuito.
2. Conecte o ESP32 ao computador pelo cabo USB.
3. Abra o arquivo `01_scanner_i2c.ino` na Arduino IDE.
4. Selecione a placa e a porta serial correspondentes ao ESP32.
5. Compile e carregue o programa.
6. Abra o monitor serial.
7. Configure o monitor serial para `115200 baud`.
8. Aguarde a execução da varredura I²C.
9. Observe os endereços apresentados no monitor serial.
10. Mantenha o teste em execução por algumas varreduras para verificar se os dois sensores continuam sendo encontrados de maneira estável.

## Resultado esperado

O monitor serial deve apresentar os dois endereços:

```text
Dispositivo encontrado no endereco 0x68
Dispositivo encontrado no endereco 0x69
Total de dispositivos encontrados: 2
```

A varredura será repetida automaticamente a cada três segundos.

## Critério de aprovação

O teste será considerado aprovado quando:

* o dispositivo `0x68` for encontrado;
* o dispositivo `0x69` for encontrado;
* somente os dois dispositivos esperados forem encontrados;
* ambos continuarem aparecendo em todas as varreduras, sem desaparecimentos intermitentes.

Se apenas um endereço aparecer, verifique principalmente a alimentação, as conexões `SDA` e `SCL` e o estado do pino `AD0` de cada sensor.


# 2. Leitura individual das IMUs

## Objetivo

Verificar individualmente o funcionamento das quatro IMUs instaladas nos dois halteres.

O teste confirma:

* a comunicação com o endereço selecionado;
* a identificação do modelo da IMU;
* a leitura dos três eixos do acelerômetro;
* a leitura dos três eixos do giroscópio;
* a leitura da temperatura interna do sensor;
* a resposta dos valores quando o halter é movimentado.

Embora as duas IMUs permaneçam conectadas ao ESP32, o firmware realiza a leitura de apenas um endereço por vez.

## Modelos compatíveis

Os módulos instalados podem se identificar como MPU-6050 ou MPU-6500. Os dois modelos possuem compatibilidade nos registradores básicos utilizados neste teste.

O modelo é identificado por meio do registrador `WHO_AM_I`:

| Resposta | Modelo identificado |
| -------- | ------------------- |
| `0x68`   | MPU-6050            |
| `0x70`   | MPU-6500            |

As duas respostas são consideradas válidas.

Também é possível que os halteres possuam uma combinação de MPU-6050 e MPU-6500. Essa situação não representa uma falha neste teste, desde que todos os sensores sejam reconhecidos e apresentem leituras válidas.

## Hardware necessário

* 1 halter do projeto HalterCheck;
* 1 ESP32 DevKit v1;
* 2 módulos inerciais instalados no halter;
* Cabos de conexão;
* Cabo USB para conectar o ESP32 ao computador.

O procedimento deve ser executado separadamente nos dois halteres.

## Montagem do hardware

A montagem utilizada no teste anterior deve ser mantida:

| ESP32 DevKit v1 | IMU A | IMU B |
| --------------- | ----- | ----- |
| 3V3             | VCC   | VCC   |
| GND             | GND   | GND   |
| GPIO 21         | SDA   | SDA   |
| GPIO 22         | SCL   | SCL   |
| GND             | AD0   | —     |
| 3V3             | —     | AD0   |

Os sensores utilizam os seguintes endereços de comunicação:

* IMU A: `0x68`;
* IMU B: `0x69`.

Os pinos `INT` não são utilizados neste teste.

## Endereço I²C e identificação do modelo

O endereço I²C e o valor do registrador `WHO_AM_I` possuem funções diferentes:

* o endereço I²C determina qual sensor receberá os comandos;
* o registrador `WHO_AM_I` identifica o modelo do sensor.

Portanto, uma IMU pode estar no endereço de comunicação `0x69` e retornar `0x68` ou `0x70` no registrador `WHO_AM_I`.

Exemplos válidos:

| Endereço I²C | `WHO_AM_I` | Interpretação               |
| ------------ | ---------- | --------------------------- |
| `0x68`       | `0x68`     | MPU-6050 no endereço `0x68` |
| `0x69`       | `0x68`     | MPU-6050 no endereço `0x69` |
| `0x68`       | `0x70`     | MPU-6500 no endereço `0x68` |
| `0x69`       | `0x70`     | MPU-6500 no endereço `0x69` |

## Firmware utilizado

Utilize o arquivo:

```text
testes_unitarios/imu_serial/02_leitura_imu_individual/02_leitura_imu_individual.ino
```

O endereço da IMU que será testada é definido pela constante:

```cpp
constexpr uint8_t MPU_ADDRESS = 0x68;
```

Para testar a IMU A, utilize:

```cpp
constexpr uint8_t MPU_ADDRESS = 0x68;
```

Para testar a IMU B, utilize:

```cpp
constexpr uint8_t MPU_ADDRESS = 0x69;
```

Após alterar o endereço, o firmware deve ser compilado e carregado novamente no ESP32.

## Configuração utilizada

O firmware configura a IMU com:

* acelerômetro na escala de `±2 g`;
* giroscópio na escala de `±250 °/s`;
* frequência interna de amostragem próxima de `50 Hz`;
* filtro digital configurado no nível 3;
* comunicação I²C em `100 kHz`;
* comunicação serial em `115200 baud`.

A exibição no monitor serial ocorre a cada 200 milissegundos para facilitar a inspeção visual. Esse intervalo de exibição não representa a frequência que será utilizada posteriormente na coleta definitiva.

## Sequência de testes

O procedimento deve ser executado quatro vezes:

1. Halter 1 — IMU A — endereço `0x68`;
2. Halter 1 — IMU B — endereço `0x69`;
3. Halter 2 — IMU A — endereço `0x68`;
4. Halter 2 — IMU B — endereço `0x69`.

## Procedimento

1. Mantenha as duas IMUs conectadas ao ESP32 conforme a montagem apresentada.
2. Conecte o halter ao computador utilizando o cabo USB.
3. Abra o arquivo `02_leitura_imu_individual.ino` na Arduino IDE.
4. Defina em `MPU_ADDRESS` o endereço da IMU que será testada.
5. Selecione a placa e a porta serial correspondentes ao ESP32.
6. Compile e carregue o firmware.
7. Abra o monitor serial.
8. Configure o monitor serial para `115200 baud`.
9. Confira o endereço selecionado.
10. Confira a resposta do registrador `WHO_AM_I`.
11. Verifique se o firmware identificou o sensor como MPU-6050 ou MPU-6500.
12. Mantenha o halter completamente parado por alguns segundos.
13. Observe os valores do acelerômetro e do giroscópio.
14. Incline lentamente o halter em diferentes direções.
15. Observe as alterações nos eixos `ax`, `ay` e `az`.
16. Rotacione o halter ao redor de seus diferentes eixos.
17. Observe as alterações nos eixos `gx`, `gy` e `gz`.
18. Repita o procedimento para o outro endereço.
19. Repita todos os passos utilizando o segundo halter.

## Inicialização esperada para um MPU-6050

```text
HalterCheck - Leitura individual do MPU-6050
--------------------------------------------
Endereco selecionado: 0x68
Resposta do registrador WHO_AM_I: 0x68
Sensor identificado: MPU-6050.
IMU identificada e configurada.
```

## Inicialização esperada para um MPU-6500

```text
HalterCheck - Leitura individual do MPU-6050
--------------------------------------------
Endereco selecionado: 0x68
Resposta do registrador WHO_AM_I: 0x70
Sensor identificado: MPU-6500.
IMU identificada e configurada.
```

Nos dois casos, o firmware deve continuar a execução e começar a apresentar as leituras.

## Formato das leituras

Após a inicialização, o firmware apresenta:

```text
ax    ay    az    temp_C    gx    gy    gz
```

Os campos possuem os seguintes significados:

| Campo    | Significado                                     |
| -------- | ----------------------------------------------- |
| `ax`     | Aceleração bruta no eixo X                      |
| `ay`     | Aceleração bruta no eixo Y                      |
| `az`     | Aceleração bruta no eixo Z                      |
| `temp_C` | Temperatura interna aproximada em graus Celsius |
| `gx`     | Velocidade angular bruta no eixo X              |
| `gy`     | Velocidade angular bruta no eixo Y              |
| `gz`     | Velocidade angular bruta no eixo Z              |

## Comportamento esperado com o halter parado

Com o halter parado:

* os valores do acelerômetro devem permanecer relativamente estáveis;
* a aceleração da gravidade deve aparecer distribuída entre os eixos;
* se um eixo estiver aproximadamente alinhado com a gravidade, sua magnitude deverá ficar próxima de `16384`;
* os valores do giroscópio devem permanecer próximos de zero;
* pequenos deslocamentos e oscilações são normais devido ao ruído e ao desvio interno do sensor;
* a temperatura deve apresentar um valor plausível e relativamente estável.

Não é necessário que o mesmo eixo apresente a gravidade nas quatro IMUs, pois isso depende da orientação física de cada sensor no halter.

## Comportamento esperado durante o movimento

Ao inclinar o halter:

* os valores de `ax`, `ay` e `az` devem mudar;
* a distribuição da gravidade entre os eixos deve acompanhar a orientação do halter.

Ao rotacionar o halter:

* pelo menos um dos valores `gx`, `gy` ou `gz` deve apresentar uma variação evidente;
* o sinal do valor deve mudar quando o sentido da rotação for invertido;
* após interromper a rotação, os valores devem retornar para uma região próxima de zero.

## Respostas não reconhecidas

Caso o registrador `WHO_AM_I` retorne um valor diferente de `0x68` e `0x70`, o firmware deve interromper o teste e apresentar:

```text
ERRO: modelo de IMU nao reconhecido.
```

Nesse caso, o sensor não deve ser considerado aprovado até que seu modelo e sua compatibilidade sejam verificados.

## Critério de aprovação

Cada IMU será considerada aprovada quando:

* responder no endereço I²C esperado;
* retornar `0x68` ou `0x70` no registrador `WHO_AM_I`;
* ser identificada como MPU-6050 ou MPU-6500;
* concluir a configuração sem mensagens de erro;
* apresentar valores em todos os eixos do acelerômetro;
* apresentar valores em todos os eixos do giroscópio;
* manter leituras relativamente estáveis quando o halter estiver parado;
* responder às mudanças de inclinação;
* responder às rotações nos diferentes eixos;
* não apresentar valores permanentemente travados;
* não apresentar falhas intermitentes de comunicação.

Não é necessário que as quatro IMUs sejam do mesmo modelo para aprovação deste teste. Caso existam modelos diferentes, essa diferença deverá ser considerada posteriormente nas etapas de calibração e normalização dos dados.

O teste completo será considerado aprovado quando as duas IMUs dos dois halteres atenderem a esses critérios.


# 3. Leitura simultânea das duas IMUs

## Objetivo

Verificar se o ESP32 consegue configurar e ler continuamente as duas IMUs instaladas em um mesmo halter.

O teste confirma:

* a comunicação simultânea com os endereços `0x68` e `0x69`;
* a identificação do modelo de cada IMU;
* a configuração das duas IMUs;
* a leitura dos acelerômetros e giroscópios;
* a organização das leituras em pares;
* a criação de uma sequência compartilhada entre as IMUs;
* a manutenção da frequência de amostragem próxima de `50 Hz`;
* a estabilidade do barramento I²C durante leituras contínuas.

O procedimento deve ser realizado separadamente nos dois halteres.

## Hardware necessário

* 1 halter do projeto HalterCheck;
* 1 ESP32 DevKit v1;
* 2 IMUs instaladas no halter;
* Cabos de conexão;
* Cabo USB para conectar o ESP32 ao computador.

## Montagem do hardware

A montagem utilizada nos testes anteriores deve ser mantida:

| ESP32 DevKit v1 | IMU A | IMU B |
| --------------- | ----- | ----- |
| 3V3             | VCC   | VCC   |
| GND             | GND   | GND   |
| GPIO 21         | SDA   | SDA   |
| GPIO 22         | SCL   | SCL   |
| GND             | AD0   | —     |
| 3V3             | —     | AD0   |

Os sensores utilizam os seguintes endereços:

* IMU A: `0x68`;
* IMU B: `0x69`.

As duas IMUs compartilham os pinos `SDA` e `SCL`, mas são acessadas separadamente por meio de seus endereços.

Os pinos `INT` não são utilizados neste teste.

## Modelos compatíveis

O firmware aceita os seguintes valores no registrador `WHO_AM_I`:

| Resposta | Modelo identificado |
| -------- | ------------------- |
| `0x68`   | MPU-6050            |
| `0x70`   | MPU-6500            |

As duas IMUs não precisam ser obrigatoriamente do mesmo modelo para a realização deste teste.

Cada sensor deve responder em seu endereço I²C e retornar um identificador compatível.

## Firmware utilizado

Utilize o arquivo:

```text
testes_unitarios/imu_serial/03_leitura_duas_imus/03_leitura_duas_imus.ino
```

O firmware utiliza os endereços:

```cpp
constexpr uint8_t IMU_A_ADDRESS = 0x68;
constexpr uint8_t IMU_B_ADDRESS = 0x69;
```

Não é necessário alterar o código para selecionar individualmente as IMUs. Os dois endereços são inicializados e lidos durante a mesma execução.

## Configuração utilizada

As duas IMUs são configuradas com:

* acelerômetro na escala de `±2 g`;
* giroscópio na escala de `±250 °/s`;
* frequência interna de amostragem próxima de `50 Hz`;
* filtro digital configurado no nível 3;
* comunicação I²C em `100 kHz`;
* comunicação serial em `115200 baud`.

O ESP32 cria um novo par de leituras a cada aproximadamente `20 ms`, correspondente à frequência-alvo de `50 Hz`.

## Inicialização das IMUs

Durante a inicialização, o firmware executa as seguintes etapas para cada sensor:

1. acessa o endereço I²C;
2. lê o registrador `WHO_AM_I`;
3. identifica o modelo da IMU;
4. retira o sensor do modo de repouso;
5. configura o filtro digital;
6. configura a frequência de amostragem;
7. configura a escala do acelerômetro;
8. configura a escala do giroscópio.

A coleta somente começa se as duas IMUs forem inicializadas corretamente.

## Procedimento

1. Confira as conexões das duas IMUs.
2. Conecte o primeiro halter ao computador utilizando o cabo USB.
3. Abra o arquivo `03_leitura_duas_imus.ino` na Arduino IDE.
4. Selecione a placa e a porta serial correspondentes ao ESP32.
5. Compile e carregue o firmware.
6. Abra o monitor serial.
7. Configure o monitor serial para `115200 baud`.
8. Confira a identificação da IMU A.
9. Confira a identificação da IMU B.
10. Verifique se a mensagem de inicialização das duas IMUs foi apresentada.
11. Mantenha o halter parado por alguns segundos.
12. Observe se as linhas dos sensores `A` e `B` são apresentadas continuamente.
13. Incline lentamente o halter em diferentes direções.
14. Observe as alterações nos acelerômetros das duas IMUs.
15. Rotacione o halter ao redor de seus diferentes eixos.
16. Observe as alterações nos giroscópios das duas IMUs.
17. Mantenha o teste em execução por alguns minutos para verificar a estabilidade da comunicação.
18. Repita todo o procedimento utilizando o segundo halter.

## Inicialização esperada

A inicialização deve apresentar informações semelhantes a:

```text
HalterCheck - Leitura simultanea das IMUs
-----------------------------------------

Inicializando IMU A no endereco 0x68
WHO_AM_I: 0x70
Modelo identificado: MPU-6500
IMU configurada.

Inicializando IMU B no endereco 0x69
WHO_AM_I: 0x68
Modelo identificado: MPU-6050
IMU configurada.

As duas IMUs foram inicializadas.
Frequencia alvo: 50 Hz por sensor.
```

Os modelos apresentados dependem do valor retornado por cada sensor.

A inicialização é válida quando ambas as IMUs são identificadas como MPU-6050 ou MPU-6500.

## Formato das leituras

Após a inicialização, o firmware apresenta o cabeçalho:

```csv
seq,t_ms,sensor,ax,ay,az,gx,gy,gz
```

Cada campo possui o seguinte significado:

| Campo    | Significado                                                         |
| -------- | ------------------------------------------------------------------- |
| `seq`    | Número compartilhado pelo par de leituras                           |
| `t_ms`   | Tempo transcorrido desde a inicialização do ESP32, em milissegundos |
| `sensor` | Identificação da IMU: `A` ou `B`                                    |
| `ax`     | Aceleração bruta no eixo X                                          |
| `ay`     | Aceleração bruta no eixo Y                                          |
| `az`     | Aceleração bruta no eixo Z                                          |
| `gx`     | Velocidade angular bruta no eixo X                                  |
| `gy`     | Velocidade angular bruta no eixo Y                                  |
| `gz`     | Velocidade angular bruta no eixo Z                                  |

A temperatura é lida internamente, mas não é incluída na saída deste teste porque não fará parte do formato principal dos dados de movimento.

## Organização em pares

Cada sequência deve produzir duas linhas consecutivas:

```csv
0,1254,A,-320,410,16220,18,-11,25
0,1254,B,280,-510,16180,21,-8,19
1,1274,A,-318,405,16218,17,-10,24
1,1274,B,285,-506,16175,20,-9,18
```

Para cada valor de `seq`:

* deve existir uma linha do sensor `A`;
* deve existir uma linha do sensor `B`;
* as duas linhas devem possuir o mesmo `t_ms`;
* a linha do sensor `A` deve aparecer antes da linha do sensor `B`.

O número de sequência começa em zero e é incrementado depois que as duas leituras são apresentadas.

## Pareamento temporal

As duas IMUs são lidas sequencialmente pelo barramento I²C, pois apenas uma comunicação pode ocorrer por vez.

O uso do mesmo `seq` e do mesmo `t_ms` indica que as duas leituras pertencem ao mesmo ciclo lógico de amostragem. Isso não significa que as duas leituras ocorreram exatamente no mesmo instante físico, mas que foram obtidas dentro do mesmo período de amostragem.

Esse pareamento será utilizado posteriormente para organizar os dados recebidos pelo servidor.

## Frequência esperada

A frequência-alvo é de `50 Hz`, portanto o intervalo esperado entre sequências é de aproximadamente:

```text
1000 ms / 50 = 20 ms
```

Os valores de `t_ms` devem apresentar um comportamento semelhante a:

```text
1254
1274
1294
1314
```

Pequenas variações ocasionais podem ocorrer devido ao tempo necessário para:

* realizar as leituras I²C;
* formatar os valores;
* transmitir o texto pela porta serial;
* executar outras operações internas do ESP32.

O comportamento deve permanecer próximo de `20 ms` por sequência, sem pausas frequentes ou prolongadas.

## Comportamento esperado com o halter parado

Com o halter parado:

* os dados dos dois sensores devem continuar sendo atualizados;
* as leituras devem permanecer relativamente estáveis;
* a gravidade deve aparecer distribuída entre os eixos do acelerômetro;
* os giroscópios devem apresentar valores próximos de zero;
* as duas IMUs podem apresentar valores diferentes devido à orientação física, ao ruído e ao desvio individual de cada sensor.

Não é esperado que os eixos das duas IMUs apresentem valores iguais, pois os sensores podem estar fixados em posições e orientações diferentes.

## Comportamento esperado durante o movimento

Ao movimentar o halter:

* os acelerômetros das duas IMUs devem responder;
* os giroscópios das duas IMUs devem responder;
* nenhum sensor deve permanecer com valores travados;
* os pares de leitura devem continuar sendo produzidos;
* a sequência deve continuar aumentando;
* não devem ocorrer falhas de comunicação.

Como as IMUs podem possuir orientações físicas diferentes, o mesmo movimento pode produzir respostas diferentes em cada eixo.

## Falhas detectadas pelo firmware

Se uma das IMUs não responder durante a inicialização, o firmware deve interromper o teste.

Exemplo:

```text
ERRO: nao foi possivel inicializar as duas IMUs.
Teste interrompido.
```

Se uma falha ocorrer durante as leituras, o firmware informa a sequência e a IMU afetada:

```text
ERRO na sequencia 152: falha na IMU B
Teste interrompido.
```

A interrupção impede que o programa continue apresentando pares incompletos como se fossem leituras válidas.

## Critério de aprovação

Cada halter será considerado aprovado quando:

* a IMU A responder no endereço `0x68`;
* a IMU B responder no endereço `0x69`;
* ambas retornarem `0x68` ou `0x70` no registrador `WHO_AM_I`;
* as duas IMUs forem configuradas sem erros;
* cada sequência possuir uma linha do sensor `A` e uma linha do sensor `B`;
* as duas linhas de cada sequência possuírem o mesmo `seq`;
* as duas linhas de cada sequência possuírem o mesmo `t_ms`;
* o intervalo entre sequências permanecer próximo de `20 ms`;
* as duas IMUs responderem aos movimentos do halter;
* nenhuma leitura permanecer permanentemente travada;
* não ocorrerem falhas intermitentes de comunicação;
* o teste permanecer estável durante alguns minutos.

O teste completo será considerado aprovado quando os dois halteres atenderem a esses critérios.


# 4. Conexão do ESP32 à rede Wi‑Fi

## Objetivo

Verificar se o ESP32 de cada halter consegue se conectar à rede Wi‑Fi que será utilizada durante os testes do HalterCheck.

O teste confirma:

* o funcionamento da interface Wi‑Fi do ESP32;
* a localização da rede configurada;
* a autenticação utilizando o SSID e a senha;
* o recebimento de um endereço IP;
* a intensidade do sinal no local do teste;
* a manutenção da conexão durante a execução.

Este teste não utiliza as IMUs e ainda não transmite dados para o computador.

## Hardware necessário

* 1 halter do projeto HalterCheck;
* 1 ESP32 DevKit v1;
* Cabo USB;
* Computador com Arduino IDE;
* Rede Wi‑Fi disponível.

As IMUs podem permanecer conectadas ao ESP32, mas não são acessadas pelo firmware deste teste.

## Requisitos da rede

A rede deve:

* estar disponível no local do teste;
* permitir a conexão do ESP32;
* utilizar a faixa de `2,4 GHz`;
* fornecer endereços IP automaticamente por DHCP;
* possuir sinal suficiente no local onde os exercícios serão realizados.

O ESP32 utilizado no projeto não se conecta diretamente a redes que operam exclusivamente em `5 GHz`.

## Firmware utilizado

Utilize o arquivo:

```text
testes_unitarios/wifi/01_conexao_wifi/01_conexao_wifi.ino
```

## Configuração do halter

O identificador do halter é definido por:

```cpp
constexpr char HALTER_ID[] = "H1";
```

Para o primeiro halter, utilize:

```cpp
constexpr char HALTER_ID[] = "H1";
```

Para o segundo halter, utilize:

```cpp
constexpr char HALTER_ID[] = "H2";
```

Neste teste, o identificador é apresentado apenas no monitor serial. Ele será incorporado às mensagens transmitidas em testes posteriores.

## Configuração da rede

Preencha o nome e a senha da rede:

```cpp
constexpr char WIFI_SSID[] = "NOME_DA_REDE";
constexpr char WIFI_PASSWORD[] = "SENHA_DA_REDE";
```

Exemplo:

```cpp
constexpr char WIFI_SSID[] = "RedeHalterCheck";
constexpr char WIFI_PASSWORD[] = "senha_da_rede";
```

As credenciais utilizadas no projeto real não devem ser publicadas em repositórios públicos.

Depois de alterar o identificador ou as credenciais, compile e carregue novamente o firmware no ESP32.

## Configurações utilizadas

O firmware utiliza:

* modo Wi‑Fi de estação;
* tempo máximo de conexão de `20 segundos`;
* apresentação do estado da conexão a cada `2 segundos`;
* comunicação serial em `115200 baud`;
* economia de energia do Wi‑Fi desativada.

A economia de energia é desativada com:

```cpp
WiFi.setSleep(false);
```

Essa configuração reduz possíveis atrasos durante a comunicação, mas aumenta o consumo de energia. A influência sobre a autonomia será analisada posteriormente no teste de alimentação por bateria.

## Procedimento

1. Posicione o halter no local onde os exercícios serão realizados.
2. Conecte o primeiro halter ao computador utilizando o cabo USB.
3. Abra o arquivo `01_conexao_wifi.ino` na Arduino IDE.
4. Defina o identificador como `H1`.
5. Preencha o nome e a senha da rede Wi‑Fi.
6. Selecione a placa e a porta serial correspondentes ao ESP32.
7. Compile e carregue o firmware.
8. Abra o monitor serial.
9. Configure o monitor serial para `115200 baud`.
10. Aguarde a tentativa de conexão.
11. Confira se o ESP32 recebeu um endereço IP.
12. Observe a intensidade do sinal apresentada em `dBm`.
13. Mantenha o teste em execução por alguns minutos.
14. Observe se o estado permanece como `conectado`.
15. Desconecte o primeiro halter do computador.
16. Conecte o segundo halter.
17. Altere o identificador para `H2`.
18. Compile e carregue o firmware.
19. Repita o procedimento para o segundo halter.

## Resultado esperado

Durante a conexão, o monitor serial deve apresentar:

```text
HalterCheck - Teste de conexao Wi-Fi
------------------------------------
Halter: H1
Rede: RedeHalterCheck
Conectando....
```

Após a conexão, devem ser apresentadas informações semelhantes a:

```text
Wi-Fi conectado.
Halter: H1
SSID: RedeHalterCheck
Endereco IP: 192.168.0.15
Gateway: 192.168.0.1
Mascara de rede: 255.255.255.0
Endereco MAC: 24:6F:28:12:34:56
Intensidade do sinal: -48 dBm
```

Os endereços apresentados dependem da configuração da rede.

Depois da inicialização, o estado deve continuar sendo exibido:

```text
Estado: conectado | RSSI: -48 dBm | IP: 192.168.0.15
```

## Endereço IP

O endereço IP identifica o ESP32 dentro da rede local.

Um endereço como:

```text
192.168.0.15
```

indica que o roteador atribuiu um endereço ao dispositivo.

Os dois halteres devem receber endereços diferentes. Por exemplo:

```text
H1: 192.168.0.15
H2: 192.168.0.16
```

Não é necessário que os endereços permaneçam iguais entre diferentes execuções, pois o roteador pode atribuir novos valores por DHCP.

## Endereço MAC

O endereço MAC identifica fisicamente a interface Wi‑Fi do ESP32.

Cada halter deve apresentar um endereço MAC diferente. Essa informação poderá ser utilizada posteriormente para auxiliar na identificação dos dispositivos, se necessário.

## Intensidade do sinal

O valor de RSSI representa a intensidade do sinal recebido. Ele é apresentado em `dBm` e normalmente possui valor negativo.

Quanto mais próximo de zero, mais forte é o sinal.

| RSSI aproximado      | Interpretação                                   |
| -------------------- | ----------------------------------------------- |
| Maior que `-50 dBm`  | Sinal forte                                     |
| De `-50` a `-67 dBm` | Sinal adequado                                  |
| De `-68` a `-75 dBm` | Sinal utilizável, sujeito a maior instabilidade |
| Menor que `-75 dBm`  | Sinal fraco para a transmissão contínua         |

Essas faixas servem como referência prática. A estabilidade da transmissão será verificada diretamente nos testes posteriores.

## Estados apresentados

O firmware pode apresentar os seguintes estados:

| Estado                             | Significado                                          |
| ---------------------------------- | ---------------------------------------------------- |
| `inicializando`                    | A interface Wi‑Fi está iniciando                     |
| `rede nao encontrada`              | O SSID configurado não foi localizado                |
| `conectado`                        | O ESP32 está conectado à rede                        |
| `falha de autenticacao ou conexao` | A conexão não foi concluída                          |
| `conexao perdida`                  | A conexão existente foi interrompida                 |
| `desconectado`                     | O ESP32 não está conectado                           |
| `estado desconhecido`              | O estado retornado não foi reconhecido pelo firmware |

## Falha na conexão inicial

Se a conexão não for estabelecida dentro de 20 segundos, o firmware deve apresentar uma mensagem semelhante a:

```text
ERRO: nao foi possivel conectar. Estado: rede nao encontrada
Teste interrompido.
```

As causas mais comuns são:

* nome da rede incorreto;
* senha incorreta;
* rede fora do alcance;
* rede disponível somente em `5 GHz`;
* sinal insuficiente;
* indisponibilidade temporária do roteador.

## Perda de conexão durante o teste

Se a conexão for perdida após a inicialização, o monitor serial deixa de apresentar o estado `conectado` e informa a condição atual.

Este firmware não tenta recuperar automaticamente uma conexão perdida. A reconexão será tratada em um teste específico posterior.

## Critério de aprovação

Cada halter será considerado aprovado quando:

* localizar a rede configurada;
* autenticar utilizando as credenciais fornecidas;
* estabelecer a conexão em até 20 segundos;
* receber um endereço IP válido;
* apresentar um endereço MAC;
* permanecer com o estado `conectado`;
* apresentar intensidade de sinal suficiente no local dos exercícios;
* manter a conexão durante alguns minutos sem interrupções.

O teste completo será considerado aprovado quando os dois halteres atenderem a esses critérios.


# 5. Reconexão automática à rede Wi‑Fi

## Objetivo

Verificar se o ESP32 consegue detectar a interrupção da conexão Wi‑Fi e restabelecê-la automaticamente, sem precisar ser reiniciado.

O teste confirma:

* a detecção de uma desconexão;
* a realização periódica de novas tentativas;
* a recuperação automática da comunicação com o ponto de acesso;
* o recebimento de um endereço IP após a reconexão;
* a repetição do processo sem travamento do ESP32.

Este teste ainda não transmite os dados das IMUs para o computador.

## Hardware necessário

* 1 halter do projeto HalterCheck;
* 1 ESP32 DevKit v1;
* Cabo USB;
* Computador com Arduino IDE;
* Rede Wi‑Fi disponível.

As IMUs podem permanecer conectadas, mas não são utilizadas pelo firmware.

## Firmware utilizado

Utilize o arquivo:

```text
testes_unitarios/wifi/02_reconexao_wifi/02_reconexao_wifi.ino
```

## Configuração do halter

Para o primeiro halter, utilize:

```cpp
constexpr char HALTER_ID[] = "H1";
```

Para o segundo halter, utilize:

```cpp
constexpr char HALTER_ID[] = "H2";
```

## Configuração da rede

Preencha o nome e a senha da rede:

```cpp
constexpr char WIFI_SSID[] = "NOME_DA_REDE";
constexpr char WIFI_PASSWORD[] = "SENHA_DA_REDE";
```

As credenciais reais não devem ser publicadas em repositórios públicos.

## Estratégia de reconexão

O firmware utiliza duas formas complementares de recuperação:

```cpp
WiFi.setAutoReconnect(true);
```

Essa configuração permite que a biblioteca Wi‑Fi tente recuperar automaticamente uma conexão perdida.

Além disso, enquanto o ESP32 estiver desconectado, o firmware executa uma nova tentativa a cada:

```cpp
constexpr uint32_t RECONNECT_INTERVAL_MS = 5000;
```

Portanto, uma nova tentativa de conexão deve ocorrer aproximadamente a cada cinco segundos.

O firmware não interrompe a execução se a conexão falhar. Ele continua tentando até que a rede volte a ficar disponível.

## Formas de provocar a desconexão

O teste pode ser executado de duas maneiras.

### Desconexão controlada pelo monitor serial

O firmware aceita o comando:

```text
D
```

Ao receber esse comando, o ESP32 encerra deliberadamente a conexão atual. Em seguida, o mecanismo de reconexão deve restabelecer a comunicação.

Esse método permite repetir o teste sem desligar o roteador.

### Indisponibilidade do ponto de acesso

Também é possível desligar temporariamente o ponto de acesso ou desativar a rede compartilhada utilizada pelo ESP32.

Nesse cenário:

1. o ESP32 detecta a perda da conexão;
2. as tentativas de reconexão continuam enquanto a rede estiver indisponível;
3. depois que a rede for reativada, o ESP32 deve se conectar novamente.

Quando possível, utilize um ponto de acesso dedicado ou o compartilhamento de internet de um celular para evitar interromper outros dispositivos da rede.

## Procedimento principal

1. Conecte o primeiro halter ao computador utilizando o cabo USB.
2. Abra o arquivo `02_reconexao_wifi.ino` na Arduino IDE.
3. Defina o identificador como `H1`.
4. Preencha o nome e a senha da rede.
5. Selecione a placa e a porta serial correspondentes ao ESP32.
6. Compile e carregue o firmware.
7. Abra o monitor serial.
8. Configure a velocidade para `115200 baud`.
9. Aguarde a conexão inicial.
10. Confirme que o estado apresentado é `conectado`.
11. Digite `D` no campo de envio do monitor serial.
12. Envie o comando.
13. Observe a interrupção da conexão.
14. Aguarde uma tentativa de reconexão.
15. Confirme que o ESP32 voltou ao estado `conectado`.
16. Repita o comando `D` pelo menos três vezes.
17. Mantenha o firmware em execução por alguns minutos.
18. Repita todo o procedimento utilizando o segundo halter e o identificador `H2`.

O final de linha do monitor serial pode ser configurado como `Nova linha` ou `Sem final de linha`, pois o firmware considera apenas o caractere `D`.

## Procedimento com indisponibilidade da rede

Depois de concluir o teste controlado:

1. mantenha o ESP32 conectado à rede;
2. desligue temporariamente o ponto de acesso;
3. observe a detecção da desconexão;
4. aguarde pelo menos duas tentativas de reconexão;
5. reative o ponto de acesso;
6. aguarde o restabelecimento da conexão;
7. confira o novo estado, o endereço IP e a intensidade do sinal.

Esse procedimento é opcional, mas representa de forma mais próxima uma interrupção real da infraestrutura de rede.

## Conexão inicial esperada

A inicialização deve apresentar:

```text
HalterCheck - Teste de reconexao Wi-Fi
-------------------------------------
Halter: H1
Digite D no monitor serial para forcar uma desconexao.

Tentando conectar a RedeHalterCheck
```

Depois da conexão:

```text
Wi-Fi conectado.
Halter: H1
Numero da conexao: 1
Endereco IP: 192.168.0.15
Endereco MAC: 24:6F:28:12:34:56
RSSI: -48 dBm
Tempo desconectado: 3241 ms
```

O número da conexão começa em `1` e é incrementado sempre que uma nova conexão é estabelecida.

## Resultado esperado após o comando D

Depois do envio do comando:

```text
Comando recebido: forcar desconexao.
```

O firmware deve detectar a alteração:

```text
Conexao Wi-Fi interrompida.
Estado: desconectado
```

Enquanto a conexão não for recuperada, devem aparecer novas tentativas:

```text
Tentativa de reconexao. Estado atual: desconectado
```

Depois da recuperação:

```text
Wi-Fi conectado.
Halter: H1
Numero da conexao: 2
Endereco IP: 192.168.0.15
Endereco MAC: 24:6F:28:12:34:56
RSSI: -49 dBm
Tempo desconectado: 5218 ms
```

A numeração deve continuar aumentando nas repetições seguintes:

```text
Numero da conexao: 3
Numero da conexao: 4
```

## Endereço IP após a reconexão

O endereço IP pode permanecer igual ou mudar após a reconexão.

As duas situações são válidas:

```text
Antes: 192.168.0.15
Depois: 192.168.0.15
```

ou:

```text
Antes: 192.168.0.15
Depois: 192.168.0.18
```

O endereço é atribuído pelo roteador. O requisito é que o ESP32 receba um endereço válido e retorne ao estado `conectado`.

O endereço MAC deve permanecer o mesmo, pois identifica a interface física do ESP32.

## Tempo desconectado

O campo:

```text
Tempo desconectado: 5218 ms
```

indica quanto tempo transcorreu entre a detecção da desconexão e o restabelecimento da conexão.

Como as tentativas programadas ocorrem a cada cinco segundos, é esperado que a recuperação controlada normalmente leve alguns segundos. Em uma indisponibilidade real, o tempo também dependerá de quando o ponto de acesso voltar a funcionar.

## Comportamento esperado durante uma interrupção prolongada

Se a rede permanecer indisponível:

* o ESP32 deve continuar executando;
* novas tentativas devem aparecer aproximadamente a cada cinco segundos;
* o dispositivo não deve reiniciar;
* o monitor serial não deve travar;
* a conexão deve ser recuperada depois que a rede voltar.

## Critério de aprovação

Cada halter será considerado aprovado quando:

* estabelecer a conexão inicial;
* detectar a desconexão provocada pelo comando `D`;
* realizar novas tentativas automaticamente;
* recuperar a conexão sem reinicialização;
* receber um endereço IP válido após a recuperação;
* manter o mesmo endereço MAC;
* repetir o processo pelo menos três vezes;
* permanecer responsivo durante as tentativas;
* recuperar a conexão depois de uma interrupção real do ponto de acesso, caso esse procedimento seja executado.

O teste completo será considerado aprovado quando os dois halteres atenderem a esses critérios.


# 6. Transmissão dos dados das IMUs por Wi‑Fi

## Objetivo

Verificar o funcionamento completo da transmissão dos dados de um halter até o computador.

Este teste integra:

* as duas IMUs;
* o barramento I²C;
* o ESP32;
* a conexão Wi‑Fi;
* a conexão TCP;
* o servidor desenvolvido em Python;
* a validação da estrutura dos dados recebidos.

O caminho percorrido pelos dados é:

```text
IMU A ─┐
       ├── ESP32 ── Wi-Fi ── TCP ── Servidor Python
IMU B ─┘
```

Neste primeiro teste, os halteres devem ser executados separadamente. A conexão simultânea dos dois halteres será verificada posteriormente.

## Arquivos utilizados

Utilize os seguintes arquivos:

```text
testes_unitarios/
└── integracao/
    ├── 01_envio_imus_wifi/
    │   └── 01_envio_imus_wifi.ino
    └── 02_servidor_tcp/
        └── servidor_tcp.py
```

O arquivo `.ino` é carregado no ESP32.

O arquivo `.py` é executado no computador que receberá os dados.

## Hardware necessário

* 1 halter do projeto HalterCheck;
* 1 ESP32 DevKit v1;
* 2 IMUs instaladas no halter;
* Cabo USB;
* Computador com Arduino IDE e Python;
* Rede Wi‑Fi disponível.

O procedimento deve ser executado separadamente nos dois halteres.

## Requisitos de software

O computador deve possuir:

* Arduino IDE;
* suporte às placas ESP32;
* Python 3.10 ou superior;
* acesso permitido à porta TCP `5000`.

O servidor utiliza apenas módulos da biblioteca padrão do Python. Não é necessário instalar pacotes adicionais.

## Funcionamento do teste

O ESP32 executa continuamente as seguintes etapas:

1. lê a IMU A;
2. lê a IMU B;
3. associa as duas leituras a uma mesma sequência;
4. cria duas linhas de dados;
5. envia as linhas ao servidor por uma conexão TCP;
6. repete o processo a aproximadamente `50 Hz`.

O servidor:

1. aguarda uma conexão TCP;
2. recebe os dados enviados pelo ESP32;
3. separa as mensagens por quebra de linha;
4. valida a quantidade e os tipos dos campos;
5. verifica o pareamento das IMUs;
6. procura sequências ausentes;
7. apresenta um resumo ao final do teste.

Este servidor é utilizado apenas para validação. Ele não salva os dados recebidos em um arquivo.

## Formato das mensagens

Cada linha possui dez campos:

```csv
halter_id,seq,t_ms,sensor,ax,ay,az,gx,gy,gz
```

| Campo       | Significado                               |
| ----------- | ----------------------------------------- |
| `halter_id` | Identificação do halter: `H1` ou `H2`     |
| `seq`       | Número compartilhado pelo par de leituras |
| `t_ms`      | Tempo do ESP32 em milissegundos           |
| `sensor`    | Identificação da IMU: `A` ou `B`          |
| `ax`        | Aceleração bruta no eixo X                |
| `ay`        | Aceleração bruta no eixo Y                |
| `az`        | Aceleração bruta no eixo Z                |
| `gx`        | Velocidade angular bruta no eixo X        |
| `gy`        | Velocidade angular bruta no eixo Y        |
| `gz`        | Velocidade angular bruta no eixo Z        |

Exemplo de um par:

```csv
H1,152,3480,A,-320,410,16220,18,-11,25
H1,152,3480,B,280,-510,16180,21,-8,19
```

Cada sequência deve possuir exatamente uma linha do sensor `A` e uma linha do sensor `B`.

## Configuração do halter

Para o primeiro halter, utilize:

```cpp
constexpr char HALTER_ID[] = "H1";
```

Para o segundo halter, utilize:

```cpp
constexpr char HALTER_ID[] = "H2";
```

Os identificadores serão enviados em todas as linhas e permitirão que o servidor diferencie a origem dos dados.

## Configuração da rede

Preencha as credenciais:

```cpp
constexpr char WIFI_SSID[] = "NOME_DA_REDE";
constexpr char WIFI_PASSWORD[] = "SENHA_DA_REDE";
```

O computador e o ESP32 devem estar conectados à mesma rede local.

## Identificação do endereço IP do computador

No Windows, abra o Prompt de Comando ou o PowerShell e execute:

```powershell
ipconfig
```

Localize o adaptador utilizado pelo computador para acessar a mesma rede do ESP32.

Procure o campo:

```text
Endereço IPv4
```

Exemplo:

```text
Endereço IPv4. . . . . . . . . . . : 192.168.0.10
```

Configure esse endereço no firmware:

```cpp
IPAddress SERVER_IP(192, 168, 0, 10);
```

Esse endereço deve pertencer ao computador que executará o servidor Python.

Não utilize:

* o endereço IP do ESP32;
* o endereço do gateway;
* o endereço de outro dispositivo;
* `0.0.0.0` no firmware.

O endereço `0.0.0.0` é utilizado apenas pelo servidor para escutar em todas as interfaces de rede do computador.

## Configuração da porta

O firmware e o servidor devem utilizar a mesma porta:

```cpp
constexpr uint16_t SERVER_PORT = 5000;
```

O servidor é iniciado com:

```powershell
python servidor_tcp.py --host 0.0.0.0 --port 5000
```

Se a porta configurada no firmware for alterada, o mesmo valor deve ser informado ao servidor.

## Firewall do Windows

Na primeira execução, o Windows pode solicitar autorização para que o Python receba conexões de rede.

Permita o acesso em redes privadas.

Se o servidor estiver em execução, mas o ESP32 não conseguir se conectar, verifique se:

* o Python foi autorizado no firewall;
* a porta `5000` está disponível;
* o computador e o ESP32 estão na mesma rede;
* o endereço IP configurado pertence ao computador.

## Ordem de execução

A ordem correta é:

1. identificar o IPv4 do computador;
2. configurar o IP no firmware;
3. configurar o identificador do halter;
4. configurar o SSID e a senha;
5. iniciar o servidor Python;
6. carregar ou reiniciar o firmware do ESP32;
7. aguardar a conexão TCP;
8. realizar o teste;
9. encerrar o servidor com `Ctrl+C`.

O servidor deve estar em execução antes que o ESP32 tente estabelecer a conexão TCP.

## Procedimento para o primeiro halter

1. Conecte o computador à rede Wi‑Fi utilizada no teste.
2. Execute `ipconfig`.
3. configure o IPv4 do computador em `SERVER_IP`;
4. configure o identificador como `H1`;
5. preencha as credenciais da rede;
6. abra um terminal na pasta `02_servidor_tcp`;
7. execute:

```powershell
python servidor_tcp.py --host 0.0.0.0 --port 5000
```

8. aguarde a mensagem:

```text
Aguardando conexão do ESP32...
```

9. carregue o firmware no primeiro halter;
10. abra o monitor serial em `115200 baud`;
11. aguarde a conexão Wi‑Fi;
12. aguarde a conexão TCP;
13. mantenha o halter parado por alguns segundos;
14. movimente e rotacione o halter;
15. mantenha a transmissão por pelo menos 30 segundos;
16. encerre o servidor utilizando `Ctrl+C`;
17. confira o resumo apresentado.

## Procedimento para o segundo halter

1. Encerre a execução do primeiro halter.
2. Altere o identificador para:

```cpp
constexpr char HALTER_ID[] = "H2";
```

3. Carregue o firmware no segundo halter.
4. Reinicie o servidor Python.
5. Repita o mesmo procedimento.
6. Confira se o servidor identifica os dados como pertencentes ao halter `H2`.

## Saída esperada no ESP32

O monitor serial deve apresentar informações semelhantes a:

```text
HalterCheck - Envio das IMUs por Wi-Fi
--------------------------------------
IMU A | endereco 0x68 | WHO_AM_I 0x70
IMU B | endereco 0x69 | WHO_AM_I 0x70
IMUs configuradas.
Tentando conectar ao Wi-Fi...
Conectando ao servidor 192.168.0.10:5000
Servidor TCP conectado.
```

Durante a transmissão, o firmware apresenta periodicamente:

```text
Wi-Fi: conectado | TCP: conectado | seq: 1250 | pares enviados: 1210 | falhas: 0
```

O valor de `seq` pode ser maior que o número de pares enviados, pois o ESP32 começa a realizar as leituras antes de concluir a conexão TCP.

## Saída esperada no servidor

Quando o ESP32 se conectar:

```text
ESP32 conectado: 192.168.0.15:52134
```

As primeiras linhas válidas são apresentadas de forma legível:

```text
H1 seq=40 t=1850 sensor=A acc=(-320,410,16220) gyro=(18,-11,25)
H1 seq=40 t=1850 sensor=B acc=(280,-510,16180) gyro=(21,-8,19)
```

Depois das primeiras linhas, o servidor apresenta atualizações periódicas:

```text
H1: 500 linhas válidas | último seq: 289
H1: 1000 linhas válidas | último seq: 539
```

Como são enviados dois registros por sequência, uma frequência de `50 Hz` produz aproximadamente:

```text
50 sequências por segundo
100 linhas por segundo
```

Em 30 segundos, o valor esperado é próximo de:

```text
1500 pares
3000 linhas
```

O total pode variar devido ao tempo necessário para estabelecer as conexões e encerrar o teste.

## Resumo esperado

Ao pressionar `Ctrl+C`, o servidor deve apresentar:

```text
Resumo do teste
----------------
Halter H1
  Linhas válidas: 3000
  Pares completos: 1500
  Pares incompletos: 0
  Sequências ausentes: 0
  Sensores duplicados: 0
  Divergências de timestamp: 0
  Reinícios de sequência: 0
Linhas inválidas: 0
```

## Interpretação do resumo

### Linhas válidas

Quantidade de linhas com dez campos e valores convertidos corretamente.

### Pares completos

Quantidade de sequências que possuem uma leitura da IMU A e uma leitura da IMU B.

### Pares incompletos

Quantidade de sequências que não possuem os dois sensores.

O valor esperado é:

```text
0
```

### Sequências ausentes

Quantidade de números de sequência que não chegaram ao servidor entre duas sequências recebidas.

O valor esperado durante uma conexão estável é:

```text
0
```

### Sensores duplicados

Indica que uma mesma sequência recebeu duas linhas identificadas como o mesmo sensor.

O valor esperado é:

```text
0
```

### Divergências de timestamp

Indica que as duas linhas de uma mesma sequência possuem valores diferentes em `t_ms`.

O valor esperado é:

```text
0
```

### Reinícios de sequência

Indica que o valor de `seq` retornou para um número menor durante a execução. Isso pode acontecer se o ESP32 for reiniciado.

Em um teste sem reinicializações, o valor esperado é:

```text
0
```

### Linhas inválidas

Quantidade de mensagens que não seguem o protocolo esperado.

O valor esperado é:

```text
0
```

## Comportamento esperado durante o movimento

Ao movimentar o halter:

* os valores dos acelerômetros devem mudar;
* os valores dos giroscópios devem mudar;
* os pares devem continuar completos;
* a sequência deve continuar aumentando;
* a conexão TCP deve permanecer ativa;
* não devem surgir linhas inválidas.

## Critério de aprovação

Cada halter será considerado aprovado quando:

* as duas IMUs forem identificadas e configuradas;
* o ESP32 se conectar à rede Wi‑Fi;
* o ESP32 se conectar ao servidor TCP;
* o servidor identificar corretamente o halter;
* os dados das duas IMUs forem recebidos;
* a transmissão permanecer ativa por pelo menos 30 segundos;
* os movimentos forem refletidos nos valores recebidos;
* todos os pares forem completos;
* não houver sequências ausentes;
* não houver sensores duplicados;
* não houver divergências de timestamp;
* não houver reinícios de sequência;
* não houver linhas inválidas;
* o firmware não apresentar falhas de transmissão.

O teste completo será considerado aprovado quando os dois halteres atenderem a esses critérios individualmente.


# 7. Funcionamento do halter alimentado pela bateria

## Objetivo

Verificar se cada halter consegue inicializar as IMUs, conectar-se à rede Wi‑Fi e transmitir dados ao servidor quando alimentado exclusivamente pela bateria.

Este teste valida a estabilidade inicial do circuito de alimentação formado por:

* bateria 18650;
* módulo carregador TP4056;
* regulador de tensão MT3608;
* ESP32;
* duas IMUs.

O teste não mede a autonomia completa da bateria. A autonomia será avaliada posteriormente em um teste de longa duração.

## Hardware necessário

* 1 halter do projeto HalterCheck;
* 1 bateria 18650;
* 1 módulo TP4056;
* 1 regulador MT3608;
* 1 ESP32 DevKit v1;
* 2 IMUs;
* Multímetro;
* Computador executando o servidor TCP;
* Rede Wi‑Fi disponível.

## Verificação da tensão

Antes de conectar a saída do regulador ao ESP32:

1. desligue o halter;
2. desconecte o cabo USB;
3. configure o multímetro para medir tensão contínua;
4. meça a tensão entre a saída positiva e a saída negativa do regulador;
5. confirme que a polaridade está correta;
6. confirme que a tensão está próxima de `5,0 V`.

A saída do regulador deve alimentar o pino `5V` ou `VIN` do ESP32, conforme a montagem adotada.

Não aplique `5 V` diretamente ao pino `3V3`.

Se a tensão estiver acima do esperado, ajuste o MT3608 antes de conectar o ESP32.

## Cuidados com as fontes de alimentação

Durante este teste, o ESP32 deve receber energia apenas pelo circuito da bateria.

Não conecte simultaneamente:

* a alimentação proveniente da bateria;
* o cabo USB do computador.

A alimentação simultânea somente deve ser utilizada se o circuito tiver sido projetado e verificado para impedir corrente reversa entre as fontes.

O teste também deve ser executado sem o TP4056 conectado ao carregador. Módulos TP4056 comuns não possuem necessariamente um circuito dedicado de compartilhamento de carga entre bateria e equipamento.

## Firmware utilizado

Utilize o mesmo firmware do teste anterior:

```text
testes_unitarios/integracao/01_envio_imus_wifi/01_envio_imus_wifi.ino
```

Para o primeiro halter, configure:

```cpp
constexpr char HALTER_ID[] = "H1";
```

Para o segundo halter, configure:

```cpp
constexpr char HALTER_ID[] = "H2";
```

As credenciais da rede, o endereço IP do computador e a porta TCP devem permanecer configurados corretamente.

## Servidor utilizado

Utilize:

```text
testes_unitarios/integracao/02_servidor_tcp/servidor_tcp.py
```

Execute no Linux com:

```bash
python3 servidor_tcp.py --host 0.0.0.0 --port 5000
```

O computador e o halter devem estar conectados à mesma rede.

## Procedimento para o primeiro halter

1. Desligue a alimentação proveniente da bateria.
2. Conecte o primeiro halter ao computador pelo USB.
3. Configure o firmware como `H1`.
4. Compile e carregue o firmware.
5. Desconecte o cabo USB.
6. Confirme que o TP4056 não está conectado ao carregador.
7. Meça a saída do regulador.
8. Confirme uma tensão próxima de `5,0 V`.
9. Inicie o servidor TCP no computador.
10. Ligue a alimentação do halter pela bateria.
11. Aguarde a conexão aparecer no servidor.
12. Confirme que o servidor identifica o dispositivo como `H1`.
13. Mantenha o halter parado por alguns segundos.
14. Movimente e rotacione o halter.
15. Mantenha a transmissão ativa durante pelo menos cinco minutos.
16. Desligue o halter.
17. Encerre o servidor com `Ctrl+C`.
18. Confira o resumo apresentado.

## Procedimento para o segundo halter

1. Desligue a alimentação proveniente da bateria.
2. Conecte o segundo halter ao computador pelo USB.
3. Configure o firmware como `H2`.
4. Compile e carregue o firmware.
5. Desconecte o cabo USB.
6. Repita a verificação da tensão.
7. Reinicie o servidor TCP.
8. Ligue o segundo halter pela bateria.
9. Confirme que o servidor identifica o dispositivo como `H2`.
10. Mantenha a transmissão ativa durante pelo menos cinco minutos.
11. Movimente o halter durante parte do teste.
12. Desligue o halter.
13. Encerre o servidor.
14. Confira o resumo.

## Resultado esperado

Mesmo sem o cabo USB, o servidor deve apresentar:

```text
ESP32 conectado: 192.168.0.15:52134
H1 seq=40 t=1850 sensor=A acc=(-320,410,16220) gyro=(18,-11,25)
H1 seq=40 t=1850 sensor=B acc=(280,-510,16180) gyro=(21,-8,19)
```

A transmissão deve continuar sendo atualizada:

```text
H1: 500 linhas válidas | último seq: 289
H1: 1000 linhas válidas | último seq: 539
```

Como o monitor serial não está disponível, o servidor é utilizado para confirmar que:

* o ESP32 inicializou;
* as IMUs foram configuradas;
* o Wi‑Fi conectou;
* o cliente TCP conectou;
* os dados estão sendo transmitidos.

## Sinais de alimentação insuficiente

Uma alimentação instável pode provocar:

* ausência completa de conexão;
* conexões TCP repetidas;
* interrupções frequentes na transmissão;
* reinicialização da sequência;
* retorno de `seq` para valores menores;
* períodos com sequências ausentes;
* funcionamento parado seguido de falha durante movimentos;
* falha quando o Wi‑Fi inicia uma transmissão.

O Wi‑Fi pode provocar variações rápidas no consumo do ESP32. Uma tensão aparentemente correta sem carga não garante estabilidade durante a transmissão.

Se houver falha, verifique:

* tensão de saída do MT3608;
* polaridade das conexões;
* estado de carga da bateria;
* capacidade de corrente do regulador;
* qualidade das soldas e conexões;
* queda de tensão durante o funcionamento;
* conexão comum de terra entre os módulos.

## Quantidade esperada de dados

Aproximadamente `100` linhas são transmitidas por segundo:

```text
50 sequências por segundo
2 sensores por sequência
100 linhas por segundo
```

Em cinco minutos, o valor esperado é próximo de:

```text
15.000 pares
30.000 linhas
```

O total pode ser ligeiramente menor devido ao tempo de conexão e ao encerramento do teste.

## Critério de aprovação

Cada halter será considerado aprovado quando:

* a saída do regulador estiver próxima de `5,0 V`;
* o ESP32 inicializar utilizando apenas a bateria;
* as duas IMUs forem inicializadas;
* o halter se conectar à rede Wi‑Fi;
* o halter se conectar ao servidor TCP;
* o servidor identificar corretamente `H1` ou `H2`;
* a transmissão permanecer ativa durante pelo menos cinco minutos;
* os movimentos forem refletidos nas leituras;
* não houver pares incompletos;
* não houver sequências ausentes;
* não houver divergências de timestamp;
* não houver linhas inválidas;
* não houver reinícios de sequência;
* não houver reconexões TCP inesperadas.

O teste completo será considerado aprovado quando os dois halteres atenderem a esses critérios individualmente.
